# Don't Block the Event Loop

This notebook covers:

1. Recognize a blocking call inside an `async def` (the single most common FastAPI production bug)
2. Push sync I/O off the event loop with `anyio.to_thread.run_sync` and `asyncio.to_thread`
3. Push CPU-bound work into a `ProcessPoolExecutor`
4. Measure latency before and after each fix

**Scope**: FastAPI + `httpx.AsyncClient` in-process. A CPU-bound stand-in (tight Python loop) and a sync-I/O stand-in (`time.sleep`).

Builds on notebook 3.1, which established that `async def` runs on the event loop and `def` runs on a threadpool. This notebook is about what happens when those rules are accidentally broken — and how to fix it without rewriting the whole handler.

## 1. The Blocking Call Bug

The disaster, in one snippet:

```python
@app.get("/quote/{ticker}")
async def get_quote(ticker: str):
    time.sleep(0.1)   # <-- blocking call inside async def
    return {"ticker": ticker, "price": 100.0}
```

`time.sleep` is a sync function. Inside `async def`, FastAPI awaits the coroutine directly on the event loop. While `time.sleep` is sleeping, **the loop is frozen** — not just for this request, for *every* request currently in flight on this worker. A single slow handler stalls the whole process.

Compare with notebook 3.1 where the *correct* form was either `def + time.sleep` (threadpool absorbs the wait) or `async def + await asyncio.sleep` (loop schedules other work during the wait). The trap is the mix: `async def` advertising "I'm cooperative", then secretly hogging the CPU.

In [ ]:
import asyncio, time
from fastapi import FastAPI
from httpx import AsyncClient, ASGITransport

LATENCY_S = 0.1

app = FastAPI()

@app.get("/blocking")
async def blocking():
    # BUG: time.sleep blocks the event loop. Every concurrent request waits in series.
    time.sleep(LATENCY_S)
    return {"mode": "blocking"}

@app.get("/cooperative")
async def cooperative():
    # CORRECT: await asyncio.sleep yields the loop.
    await asyncio.sleep(LATENCY_S)
    return {"mode": "cooperative"}

print("two handlers registered: /blocking (the bug), /cooperative (the right way)")

## 2. Watching the Bug Happen

We'll fire 10 concurrent requests at each endpoint. If concurrency works, total time should be close to `LATENCY_S` (one wait). If the loop is blocked, total time will be close to `10 * LATENCY_S` (serialized waits).

In [ ]:
N = 10

async def bench(path: str, n: int = N) -> float:
    async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as ac:
        # warm-up at full concurrency so threadpool/process-pool workers are all hot
        await asyncio.gather(*(ac.get(path) for _ in range(n)))
        start = time.perf_counter()
        await asyncio.gather(*(ac.get(path) for _ in range(n)))
        return time.perf_counter() - start

async def show():
    for path in ["/blocking", "/cooperative"]:
        t = await bench(path)
        print(f"{path:14} {N} concurrent -> {t*1000:7.1f} ms  (serial would be {N*LATENCY_S*1000:.0f} ms; concurrent would be {LATENCY_S*1000:.0f} ms)")

asyncio.run(show())

You should see `/blocking` near `N * LATENCY_S` (the loop is frozen — each request waits for the previous one to release it) while `/cooperative` finishes near a single `LATENCY_S`.

This is **not** a slow handler problem. It's a *correctness* problem: the blocking endpoint also stalls every *other* in-flight request on the worker — health checks, metrics, other users — for the duration of the blocking call. In a real service that handler will get blamed for incidents it didn't cause.

## 3. How to Spot Blocking Calls in Review

Inside `async def`, every line should be one of:

- **`await something`** — cooperative.
- **Pure Python** that's clearly fast (dict access, arithmetic, simple Pydantic construction).
- **An offload primitive**: `await asyncio.to_thread(...)`, `await anyio.to_thread.run_sync(...)`, or `loop.run_in_executor(...)`.

Anything else is suspect. The usual suspects:

| Library    | Blocking call         | Async alternative              |
|------------|-----------------------|--------------------------------|
| `requests` | `requests.get(...)`   | `httpx.AsyncClient` + `await`  |
| `psycopg2` | `cursor.execute(...)` | `asyncpg` or `psycopg` v3 async |
| `redis`    | `r.get(...)`          | `redis.asyncio`                |
| `time`     | `time.sleep(...)`     | `await asyncio.sleep(...)`     |
| `open`     | `f.read()` on disk    | `aiofiles`, or `to_thread`     |

If the library has no async version (legacy code, niche driver), the fallback is to offload it — covered next.

## 4. Fix #1: `asyncio.to_thread` (stdlib, since 3.9)

`asyncio.to_thread(fn, *args)` runs `fn` in the default executor (a `ThreadPoolExecutor`) and returns a coroutine you can `await`. The handler stays `async def`; the slow sync call gets offloaded.

This is the right fix when the sync function does **I/O** (network, disk, sleep) — the GIL is released during the wait, so the offloaded thread doesn't fight the event loop for CPU.

In [ ]:
@app.get("/fixed-to-thread")
async def fixed_to_thread():
    # offload the sync call so the loop stays free
    await asyncio.to_thread(time.sleep, LATENCY_S)
    return {"mode": "fixed-to-thread"}

async def show():
    t = await bench("/fixed-to-thread")
    print(f"/fixed-to-thread {N} concurrent -> {t*1000:7.1f} ms  (matches /cooperative)")

asyncio.run(show())

## 5. Fix #2: `anyio.to_thread.run_sync` (what FastAPI uses internally)

`anyio` is the async-runtime abstraction Starlette/FastAPI is built on. Its `to_thread.run_sync` is functionally equivalent to `asyncio.to_thread` here, but with two perks worth knowing:

- **Limiter support**: you can cap how many threads a particular kind of work can occupy, preventing one slow downstream from starving the pool that serves your sync routes.
- **Cancellation semantics**: integrates with anyio task groups for structured cancellation.

In day-to-day code either form is fine. FastAPI codebases tend to lean on anyio because it's already a dependency.

In [ ]:
import anyio

@app.get("/fixed-anyio")
async def fixed_anyio():
    await anyio.to_thread.run_sync(time.sleep, LATENCY_S)
    return {"mode": "fixed-anyio"}

async def show():
    t = await bench("/fixed-anyio")
    print(f"/fixed-anyio    {N} concurrent -> {t*1000:7.1f} ms  (same shape as /fixed-to-thread)")

asyncio.run(show())

## 6. CPU-Bound Work: Threads Aren't Enough

Threads help for **I/O-bound** sync calls because the GIL is released while waiting on the OS. They do *not* help for **CPU-bound** Python because the GIL is held the entire time — a second thread doing CPU work just trades cycles with the first.

We can see this with a tight Python loop. Let's compute a fake portfolio risk score that's just a sum of squares — pure Python, no library that releases the GIL:

A note before the code: a `ProcessPoolExecutor` later in this notebook needs `risk_score` to live in an **importable module**, because worker processes look the function up by qualified name. In a production FastAPI app this is automatic — your CPU function would live in `services/risk.py`, not inline in `app.py`. Here we mirror that by writing a one-function sidecar module to disk and importing from it.

In [ ]:
from pathlib import Path

Path("_risk_module.py").write_text('''
def risk_score(n_iters=1_000_000):
    # CPU-bound stand-in for, say, a Monte Carlo VaR calculation.
    total = 0
    for i in range(n_iters):
        total += i * i
    return total
''')

import importlib
import _risk_module
importlib.reload(_risk_module)
from _risk_module import risk_score

# Calibrate so one call takes ~100ms on a typical machine. Adjust n_iters if needed.
t0 = time.perf_counter()
risk_score()
print(f"one risk_score() call: {(time.perf_counter()-t0)*1000:.1f} ms")

In [ ]:
@app.get("/risk-blocking")
async def risk_blocking():
    # BUG: CPU-bound Python in async def -> blocks loop for the full compute.
    return {"score": risk_score()}

@app.get("/risk-threaded")
async def risk_threaded():
    # Offloaded to a thread. Loop is freed... but GIL still serializes the work.
    score = await asyncio.to_thread(risk_score)
    return {"score": score}

async def show():
    for path in ["/risk-blocking", "/risk-threaded"]:
        t = await bench(path, n=4)
        print(f"{path:18} 4 concurrent -> {t*1000:7.1f} ms")

asyncio.run(show())

Two observations:

- `/risk-blocking` runs all 4 calls serially because the loop is frozen during each compute. Total ≈ `4 * one_call_ms`.
- `/risk-threaded` runs them on threads, but the GIL still serializes Python bytecode. Total ≈ `4 * one_call_ms` too — *the loop is no longer blocked* (other endpoints would stay responsive), but throughput on `/risk-threaded` itself doesn't improve.

The first point is what matters in production: the threaded version doesn't stall every other in-flight request. That alone is reason enough to offload. But if you also need *throughput* for CPU-bound work, threads can't deliver it. Processes can.

## 7. Fix #3: `ProcessPoolExecutor` for CPU-Bound

A `ProcessPoolExecutor` spawns separate Python processes. Each has its own GIL, so multiple CPU-bound calls actually run in parallel — limited only by core count.

Cost: the function and its arguments must be picklable, and inter-process calls have setup overhead (typically a few ms per call). Worth it for ≥10-50ms compute; overkill for microsecond work.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

# A module-level pool, reused across requests. Real apps create this in a lifespan startup hook
# (notebook 8.1) and shut it down on exit.
pool = ProcessPoolExecutor(max_workers=4)

@app.get("/risk-process")
async def risk_process():
    loop = asyncio.get_running_loop()
    # run_in_executor schedules the call on the process pool and returns an awaitable Future.
    score = await loop.run_in_executor(pool, risk_score)
    return {"score": score}

async def show():
    t = await bench("/risk-process", n=4)
    print(f"/risk-process     4 concurrent -> {t*1000:7.1f} ms  (parallel across CPU cores)")

asyncio.run(show())

pool.shutdown(wait=True)

On a multi-core machine `/risk-process` should finish ~`one_call_ms` (perfectly parallel) plus a few ms of pickling overhead. The threaded version was bounded by the GIL; the process pool isn't.

When to graduate further:

- **Many small CPU jobs** that still need parallelism — process pool stays fine.
- **Very heavy compute** (seconds-to-minutes per job) — push to a real worker (Celery, RQ, SQS consumer). The handler enqueues a job ID; the client polls or receives a webhook. See notebook 3.3 for the lightweight version of this with `BackgroundTasks`.

## Key Takeaways

- **The bug**: any blocking call inside `async def` freezes the event loop for every concurrent request, not just the offending one.
- **Quick audit**: inside `async def`, every line should be `await`, fast pure Python, or an explicit offload primitive.
- **I/O-bound sync** → `await asyncio.to_thread(fn, ...)` or `await anyio.to_thread.run_sync(fn, ...)`. Both are fine; pick the convention your codebase already uses.
- **CPU-bound sync** → `ProcessPoolExecutor` via `loop.run_in_executor(pool, fn)`. Threads release the loop but the GIL still serializes Python bytecode.
- **Very heavy compute** → graduate to a real worker queue. The handler should enqueue, not execute.
- **Loop-freed ≠ faster**: offloading CPU work to threads keeps the loop responsive but doesn't increase that endpoint's throughput. Only processes do.

## Exercises

**1. Spot the bug.** For each snippet, decide whether it blocks the event loop. If yes, write the one-line fix.

```python
# (a)
@app.get("/a")
async def a():
    return {"data": requests.get("https://api.example.com/x").json()}

# (b)
@app.get("/b")
async def b():
    with open("/var/log/big.log") as f:
        return {"lines": len(f.readlines())}

# (c)
@app.get("/c")
async def c():
    return {"x": sum(range(1_000))}

# (d)
@app.get("/d")
async def d():
    rows = await db.fetch("SELECT * FROM assets")  # asyncpg
    return {"n": len(rows)}
```

**2. Offload a real sync library.** Take a sync call (e.g., `time.sleep(0.05)` standing in for `requests.get`) and write three versions: blocking-in-async, `asyncio.to_thread`, and `anyio.to_thread.run_sync`. Benchmark each at `N=20`. Confirm the latter two have identical shape.

**3. CPU-bound parallelism.** Replace the `risk_score` thread-pool version with a process-pool version. Hit it with `N=4`. Show the total wall time drops from ~`4 * one_call` (GIL-serialized) to ~`one_call` (CPU-parallel) on a 4-core machine.